# Clean Atlas registrations and AFDC stations to ZIP level

**Inputs** (in `data/raw/`, never edited): the Atlas EV registrations CSV and the AFDC alternative fuel stations CSV.
**Outputs** (in `data/processed/`): `atlas_zip.csv`, `afdc_zip.csv`, and `cleaning_log.json` (the counts to copy into the provenance log and data dictionary).

Every filter is printed, so the numbers you cite in the writeup come from this notebook. Run top to bottom.

## 0. Setup and paths

In [3]:
from pathlib import Path
import json
import tempfile
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

def find_root(start=None):
    """Walk up from the working directory until a folder containing data/ is found."""
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / "data").is_dir():
            return p
    raise FileNotFoundError(f"No data/ folder found at or above {start}")

ROOT = find_root()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

def pick(pattern):
    matches = sorted(RAW.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matching {pattern} in {RAW}")
    if len(matches) > 1:
        print(f"WARNING: several files match {pattern}; using {matches[0].name}: {[m.name for m in matches]}")
    return matches[0]

ATLAS_RAW = pick("CO_EV_Registrations*.csv")
AFDC_RAW = pick("alt_fuel_stations*.csv")
LOG = {"atlas_file": ATLAS_RAW.name, "afdc_file": AFDC_RAW.name}

print("repo root:", ROOT)
print("Atlas file:", ATLAS_RAW.name)
print("AFDC file: ", AFDC_RAW.name)

repo root: c:\Users\eddie\IdeaProjects\Colorado-EV-Charging-Analysis-1
Atlas file: CO_EV_Registrations_10_2026.csv
AFDC file:  alt_fuel_stations (Oct 15 2026).csv


## 1. Atlas registrations

**Unit of observation (raw):** one row is a group of vehicles with the same ZIP, registration date, make, model, model year, drivetrain, weight class and DMV snapshot. `Vehicle Count` is the number of vehicles in the group, so sum it; do not count rows.
The file stacks monthly DMV snapshots, so we keep **only the latest snapshot**. Otherwise the same vehicles are counted once per month.

In [4]:
def read_atlas(path):
    """Read the Atlas CSV. If the download was truncated mid-row, drop the broken last line
    in a temporary copy (the raw file is untouched) and say so. Re-downloading is better."""
    try:
        return pd.read_csv(path, dtype=str, low_memory=False), False
    except pd.errors.ParserError as err:
        print("ParserError on first read:", str(err)[:120])
        print("The file is probably truncated. Dropping the final partial line in a temp copy.")
        with open(path, encoding="utf-8", newline="") as src:
            lines = src.readlines()
        tmp = Path(tempfile.gettempdir()) / "atlas_repaired.csv"
        with open(tmp, "w", encoding="utf-8", newline="") as dst:
            dst.writelines(lines[:-1])
        return pd.read_csv(tmp, dtype=str, low_memory=False), True

atlas, repaired = read_atlas(ATLAS_RAW)
LOG["atlas_repaired_truncated_tail"] = repaired

needed = ["ZIP Code", "Registration Date", "Vehicle Model Year", "Drivetrain Type",
          "Vehicle GVWR Category", "Vehicle Count", "DMV Snapshot (Date)", "Latest DMV Snapshot Flag"]
missing_cols = [c for c in needed if c not in atlas.columns]
assert not missing_cols, f"Atlas file is missing expected columns: {missing_cols}"

print("raw rows:", len(atlas))
print("columns:", atlas.columns.tolist())
LOG["atlas_raw_rows"] = len(atlas)

raw rows: 5970397
columns: ['State', 'ZIP Code', 'Registration Date', 'Vehicle Make', 'Vehicle Model', 'Vehicle Model Year', 'Drivetrain Type', 'Vehicle GVWR Class', 'Vehicle GVWR Category', 'Vehicle Count', 'DMV Snapshot ID', 'DMV Snapshot (Date)', 'Latest DMV Snapshot Flag']


In [5]:
# Parse types
atlas["Vehicle Count"] = pd.to_numeric(atlas["Vehicle Count"], errors="coerce")
atlas["snap"] = pd.to_datetime(
    atlas["DMV Snapshot (Date)"].str.extract(r"\((.*)\)")[0], format="%m/%d/%Y", errors="coerce")
atlas["reg_date"] = pd.to_datetime(atlas["Registration Date"], errors="coerce")

print("missing/non-numeric Vehicle Count:", atlas["Vehicle Count"].isna().sum())
print("unparsed snapshot dates:         ", atlas["snap"].isna().sum())
print("unparsed registration dates:     ", atlas["reg_date"].isna().sum())

atlas = atlas.dropna(subset=["Vehicle Count", "snap"]).copy()
atlas["Vehicle Count"] = atlas["Vehicle Count"].astype(int)

# Registration-date window per snapshot (documents how the date field behaves)
win = atlas.groupby("snap")["reg_date"].agg(["min", "max"])
win["window_months"] = ((win["max"] - win["min"]).dt.days / 30.4).round(1)
print("\nSnapshots in file:", len(win))
win.tail(8)

missing/non-numeric Vehicle Count: 0
unparsed snapshot dates:          0
unparsed registration dates:      0

Snapshots in file: 90


,min,max,window_months
snap,,,
2025-12-07,2016-09-01,2025-12-01,111.1
2026-01-07,2016-09-01,2026-01-01,112.1
2026-02-07,2016-09-01,2026-02-01,113.2
2026-03-07,2016-09-01,2026-03-01,114.1
2026-04-07,2016-09-01,2026-04-01,115.1
2026-05-07,2016-09-01,2026-05-01,116.1
2026-06-07,2016-09-01,2026-06-01,117.1
2026-07-07,2016-09-01,2026-07-01,118.1


In [6]:
# Keep the latest snapshot only
latest_date = atlas["snap"].max()
latest = atlas[atlas["snap"] == latest_date].copy()

flag_dates = atlas.loc[atlas["Latest DMV Snapshot Flag"].astype(str).str.lower() == "true", "snap"].unique()
print("latest snapshot by date:", latest_date.date())
print("snapshot(s) carrying the 'latest' flag:", [pd.Timestamp(d).date() for d in flag_dates])
print("rows in latest snapshot:", len(latest), "| vehicles:", latest["Vehicle Count"].sum())
print("registration dates in latest snapshot:", latest["reg_date"].min().date(), "to", latest["reg_date"].max().date())
print("oldest model years present:", sorted(latest["Vehicle Model Year"].dropna().unique())[:6])

print("\nDrivetrain types in latest snapshot:")
print(latest["Drivetrain Type"].value_counts(dropna=False))
print("\nWeight classes in latest snapshot (vehicles):")
print(latest.groupby("Vehicle GVWR Category", dropna=False)["Vehicle Count"].sum())

LOG.update({
    "atlas_latest_snapshot": str(latest_date.date()),
    "atlas_latest_snapshot_vehicles_all_classes": int(latest["Vehicle Count"].sum()),
})

latest snapshot by date: 2026-07-07
snapshot(s) carrying the 'latest' flag: [datetime.date(2026, 7, 7)]
rows in latest snapshot: 161053 | vehicles: 215105
registration dates in latest snapshot: 2016-09-01 to 2026-07-01
oldest model years present: ['1990', '1991', '1992', '1993', '1994', '1996']

Drivetrain types in latest snapshot:
Drivetrain Type
BEV     115207
PHEV     45846
Name: count, dtype: int64

Weight classes in latest snapshot (vehicles):
Vehicle GVWR Category
Heavy-Duty Vehicle (Class 7-8)           141
Light-Duty (Class 1-2A)               202639
Medium-Duty Vehicle (Class 2B - 6)     10685
Unknown                                 1640
Name: Vehicle Count, dtype: int64


In [7]:
# Light-duty only (the research question is about residential adoption)
ld = latest[latest["Vehicle GVWR Category"].str.startswith("Light-Duty", na=False)].copy()
print("light-duty vehicles:", ld["Vehicle Count"].sum(),
      "| removed (other classes):", latest["Vehicle Count"].sum() - ld["Vehicle Count"].sum())

# ZIPs as 5-character strings
ld["zip"] = ld["ZIP Code"].astype(str).str.strip().str[:5]
bad_zip = ~ld["zip"].str.fullmatch(r"\d{5}")
print("rows with invalid ZIP (dropped):", bad_zip.sum(), "| vehicles:", ld.loc[bad_zip, "Vehicle Count"].sum())
ld = ld[~bad_zip]

# One row per ZIP
piv = ld.pivot_table(index="zip", columns="Drivetrain Type", values="Vehicle Count",
                     aggfunc="sum", fill_value=0)
for c in ["BEV", "PHEV"]:
    if c not in piv.columns:
        piv[c] = 0
other = [c for c in piv.columns if c not in ("BEV", "PHEV")]
if other:
    print("NOTE: other drivetrain types found, combined into other_ev:", other)

atlas_zip = piv[["BEV", "PHEV"]].copy()
atlas_zip["other_ev"] = piv[other].sum(axis=1) if other else 0
atlas_zip["total_ev"] = atlas_zip[["BEV", "PHEV", "other_ev"]].sum(axis=1)
atlas_zip = atlas_zip.astype(int).reset_index()
atlas_zip.columns.name = None

# The table must account for every light-duty vehicle that survived the ZIP check
assert atlas_zip["total_ev"].sum() == ld["Vehicle Count"].sum(), "ZIP table does not match latest snapshot"
print("OK: ZIP table total equals light-duty vehicles in the latest snapshot:", atlas_zip["total_ev"].sum())

co_pattern = atlas_zip["zip"].str.match(r"^8[01]\d{3}$")
print("ZIPs:", len(atlas_zip), "| Colorado-pattern ZIPs (80xxx/81xxx):", co_pattern.sum(),
      "| other ZIPs:", (~co_pattern).sum(),
      "| vehicles in other ZIPs:", atlas_zip.loc[~co_pattern, "total_ev"].sum())

atlas_zip.to_csv(PROC / "atlas_zip.csv", index=False)
LOG.update({
    "atlas_light_duty_vehicles": int(atlas_zip["total_ev"].sum()),
    "atlas_zip_rows": int(len(atlas_zip)),
    "atlas_colorado_pattern_zips": int(co_pattern.sum()),
    "atlas_vehicles_outside_colorado_pattern_zips": int(atlas_zip.loc[~co_pattern, "total_ev"].sum()),
})
atlas_zip.sort_values("total_ev", ascending=False).head(10)

light-duty vehicles: 202639 | removed (other classes): 12466
rows with invalid ZIP (dropped): 0 | vehicles: 0
OK: ZIP table total equals light-duty vehicles in the latest snapshot: 202639
ZIPs: 1074 | Colorado-pattern ZIPs (80xxx/81xxx): 485 | other ZIPs: 589 | vehicles in other ZIPs: 846


,zip,BEV,PHEV,other_ev,total_ev
385,80134,4071,1089,0,5160
340,80016,3590,1005,0,4595
430,80249,2487,1930,0,4417
377,80126,1968,1693,0,3661
505,80516,2484,682,0,3166
350,80027,2478,666,0,3144
425,80238,2452,665,0,3117
362,80108,2193,737,0,2930
339,80015,2298,572,0,2870
498,80504,2065,650,0,2715


**Atlas checks to read before moving on**
- Compare `atlas_light_duty_vehicles` with the EVs-on-the-road figure on the EValuateCO Home tab for the same snapshot (check that the vehicle scope matches).
- ZIP 80249 has more PHEVs than BEVs, unlike other large ZIPs. Decide whether to keep, flag or exclude it, and document the choice.
- Many ZIPs have only a handful of EVs. Rates computed later need a minimum-household cutoff.

## 2. AFDC stations

**Unit of observation (raw):** one row per station location. Port counts are in `EV Level1 EVSE Num`, `EV Level2 EVSE Num` and `EV DC Fast Count`; a blank means that charger type is not present or not reported.
**Filter:** status = open (`E`) and access = public. This drops temporarily unavailable (`T`), planned (`P`) and private stations.

In [8]:
afdc = pd.read_csv(AFDC_RAW, dtype={"ZIP": str}, low_memory=False)
print("raw rows:", len(afdc))
print("fuel types:", afdc["Fuel Type Code"].unique().tolist(), "| states:", afdc["State"].unique().tolist())
print("duplicate station IDs:", afdc["ID"].duplicated().sum())
print("\nStatus codes:\n", afdc["Status Code"].value_counts(dropna=False))
print("\nAccess codes:\n", afdc["Access Code"].value_counts(dropna=False))

LOG.update({
    "afdc_raw_rows": int(len(afdc)),
    "afdc_status_counts": afdc["Status Code"].value_counts(dropna=False).to_dict(),
    "afdc_access_counts": afdc["Access Code"].value_counts(dropna=False).to_dict(),
})

raw rows: 3168
fuel types: ['ELEC'] | states: ['CO']
duplicate station IDs: 0

Status codes:
 Status Code
E    3104
T      52
P      12
Name: count, dtype: int64

Access codes:
 Access Code
public     2972
private     196
Name: count, dtype: int64


In [9]:
ports = ["EV Level1 EVSE Num", "EV Level2 EVSE Num", "EV DC Fast Count"]

op = afdc[(afdc["Status Code"] == "E") & (afdc["Access Code"] == "public")].copy()
print("open + public stations:", len(op))

# ZIPs
op["zip"] = op["ZIP"].astype(str).str.strip().str[:5]
bad = ~op["zip"].str.fullmatch(r"\d{5}")
print("invalid ZIPs:", bad.sum())
op = op[~bad]

# Missing port counts: blank = charger type not present; check no station is blank everywhere
print("stations with no port counts at all:", op[ports].isna().all(axis=1).sum())
op[ports] = op[ports].fillna(0)
op["total_ports"] = op[ports].sum(axis=1)

# Possible duplicate sites (same name and address, different IDs): review, do not drop automatically
dups = op[op.duplicated(["Station Name", "Street Address", "City"], keep=False)]
print("rows sharing name + address with another row:", len(dups))
dups[["ID", "Station Name", "Street Address"] + ports].sort_values("Street Address")

open + public stations: 2909
invalid ZIPs: 0
stations with no port counts at all: 0
rows sharing name + address with another row: 8


,ID,Station Name,Street Address,EV Level1 EVSE Num,EV Level2 EVSE Num,EV DC Fast Count
100,82165,Pueblo County Courthouse,215 W 10th St,0.0,2.0,0.0
2475,417725,Pueblo County Courthouse,215 W 10th St,0.0,6.0,0.0
2699,449837,National Western Complex,3625 E 48th Ave,0.0,0.0,4.0
3052,465937,National Western Complex,3625 E 48th Ave,0.0,0.0,2.0
2354,397443,"CircleK - Broomfield, CO",425 W Highway 287,0.0,0.0,3.0
2355,397444,"CircleK - Broomfield, CO",425 W Highway 287,0.0,0.0,1.0
388,157972,Fairfield Inn & Suites by Marriott Denver Tech...,5071 S Syracuse St,0.0,1.0,0.0
389,157973,Fairfield Inn & Suites by Marriott Denver Tech...,5071 S Syracuse St,0.0,1.0,0.0


In [10]:
# Largest sites: legitimate, but a few big (often apartment) sites can dominate a ZIP's total
cols = ["Station Name", "City", "ZIP", "EV Network", "Access Detail Code", "Restricted Access", "Facility Type", "total_ports"]
print(op.nlargest(12, "total_ports")[cols].to_string())
print("\nAccess detail codes:\n", op["Access Detail Code"].value_counts(dropna=False).head(8))
print("\nRestricted access:\n", op["Restricted Access"].value_counts(dropna=False).head(8))

confirmed = pd.to_datetime(op["Date Last Confirmed"], errors="coerce")
print("\nstations not confirmed since 2024-01-01:", (confirmed < "2024-01-01").sum())
print("\ntop networks:\n", op["EV Network"].value_counts(dropna=False).head(8))

                                               Station Name              City    ZIP         EV Network  Access Detail Code Restricted Access        Facility Type  total_ports
2429                            Camry Parking Lot - EV Only            Denver  80204              FLASH                 NaN             False               PUBLIC         61.0
2390                                     AMLI Broadway Park            Denver  80223               LOOP  CREDIT_CARD_ALWAYS             False                OTHER         59.0
213                Fine Airport Parking - Tesla Destination            Aurora  80019  Tesla Destination                 NaN             False       PARKING_GARAGE         48.0
2776                                      AMLI Art District            Denver  80216               LOOP  CREDIT_CARD_ALWAYS             False                OTHER         41.0
2778                                         AMLI Ridgegate         Lone Tree  80124               LOOP  CREDIT_CARD_ALW

In [11]:
afdc_zip = (op.groupby("zip")
              .agg(stations=("ID", "count"),
                   l1_ports=("EV Level1 EVSE Num", "sum"),
                   l2_ports=("EV Level2 EVSE Num", "sum"),
                   dcfc_ports=("EV DC Fast Count", "sum"),
                   networks=("EV Network", "nunique"))
              .reset_index())
afdc_zip[["l1_ports", "l2_ports", "dcfc_ports"]] = afdc_zip[["l1_ports", "l2_ports", "dcfc_ports"]].astype(int)
afdc_zip["total_ports"] = afdc_zip[["l1_ports", "l2_ports", "dcfc_ports"]].sum(axis=1)

# The ZIP table must account for every station and port that survived the filters
assert afdc_zip["stations"].sum() == len(op)
assert afdc_zip["total_ports"].sum() == op["total_ports"].sum()
print("OK: ZIP table accounts for all", len(op), "stations and", afdc_zip["total_ports"].sum(), "ports")

print("ZIPs with a public station:", len(afdc_zip))
print("Level 2 ports:", afdc_zip["l2_ports"].sum(), "| DC fast ports:", afdc_zip["dcfc_ports"].sum(),
      "| Level 1 ports:", afdc_zip["l1_ports"].sum())

afdc_zip.to_csv(PROC / "afdc_zip.csv", index=False)
LOG.update({
    "afdc_open_public_stations": int(len(op)),
    "afdc_zip_rows": int(len(afdc_zip)),
    "afdc_l1_ports": int(afdc_zip["l1_ports"].sum()),
    "afdc_l2_ports": int(afdc_zip["l2_ports"].sum()),
    "afdc_dcfc_ports": int(afdc_zip["dcfc_ports"].sum()),
})
afdc_zip.sort_values("total_ports", ascending=False).head(10)

OK: ZIP table accounts for all 2909 stations and 7924 ports
ZIPs with a public station: 294
Level 2 ports: 6161 | DC fast ports: 1708 | Level 1 ports: 55


,zip,stations,l1_ports,l2_ports,dcfc_ports,networks,total_ports
55,80202,111,0,308,4,9,312
98,80401,84,0,168,44,9,212
92,80301,80,0,165,29,8,194
57,80204,73,0,189,2,6,191
90,80249,44,37,80,57,11,174
33,80112,45,0,130,42,11,172
153,80538,48,0,143,27,10,170
67,80216,25,0,150,6,8,156
103,80424,61,0,130,2,4,132
93,80302,58,0,123,3,7,126


**AFDC cross-check (optional).** If you downloaded the EValuateCO DIC ports file, its port totals come from AFDC and should be close to these (earlier comparison: about 6,079 Level 2 and 1,705 DC fast there, versus roughly 6,161 and 1,708 for open + public here; differences are likely snapshot timing).

## 3. Save the cleaning log

In [12]:
with open(PROC / "cleaning_log.json", "w") as f:
    json.dump(LOG, f, indent=2, default=str)
print(json.dumps(LOG, indent=2, default=str))
print("\nWrote:", [p.name for p in sorted(PROC.glob("*")) if p.name in ("atlas_zip.csv", "afdc_zip.csv", "cleaning_log.json")])

{
  "atlas_file": "CO_EV_Registrations_10_2026.csv",
  "afdc_file": "alt_fuel_stations (Oct 15 2026).csv",
  "atlas_repaired_truncated_tail": false,
  "atlas_raw_rows": 5970397,
  "atlas_latest_snapshot": "2026-07-07",
  "atlas_latest_snapshot_vehicles_all_classes": 215105,
  "atlas_light_duty_vehicles": 202639,
  "atlas_zip_rows": 1074,
  "atlas_colorado_pattern_zips": 485,
  "atlas_vehicles_outside_colorado_pattern_zips": 846,
  "afdc_raw_rows": 3168,
  "afdc_status_counts": {
    "E": 3104,
    "T": 52,
    "P": 12
  },
  "afdc_access_counts": {
    "public": 2972,
    "private": 196
  },
  "afdc_open_public_stations": 2909,
  "afdc_zip_rows": 294,
  "afdc_l1_ports": 55,
  "afdc_l2_ports": 6161,
  "afdc_dcfc_ports": 1708
}

Wrote: ['afdc_zip.csv', 'atlas_zip.csv', 'cleaning_log.json']
